# Scene 2 Full-Frame Experimental Gemini Realization — Prompt-Normalized

This notebook keeps the **working structure** of your experimental full-frame notebook and changes only the realization step from **OpenAI** to **Gemini image editing**.

## What changed
- Keeps the same repo-based pipeline, Florence parsing, SAM2, Depth Anything, scene representation, and layer planning.
- Replaces the `openai_edit(...)` realization call with a local `gemini_edit(...)` helper.
- Keeps the same stronger per-layer normalization prompt.
- Resizes Gemini output back to the original full-frame canvas size when needed.

## Important note
Gemini's native image editing flow accepts the base image plus a text instruction. It does **not** use the same explicit PNG mask edit endpoint pattern as the OpenAI notebook, so this version preserves the existing **focused full-frame input** approach and uses prompt constraints to keep the edit targeted.


In [ ]:
%cd /content
!git clone https://github.com/UrbinaDan/PaperTheater_ai_Pipeline.git
%cd PaperTheater_ai_Pipeline

!pip install -q numpy scipy matplotlib opencv-python-headless pillow shapely svgwrite cairosvg tqdm networkx pyyaml requests google-genai accelerate bitsandbytes einops sentencepiece transformers==4.49.0

In [ ]:
import os
from getpass import getpass
os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

In [ ]:
%cd /content
!git clone https://github.com/facebookresearch/sam2.git
%cd sam2
!pip install -e .
!mkdir -p checkpoints
!wget -q -P checkpoints https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt
!wget -q -P checkpoints https://raw.githubusercontent.com/facebookresearch/sam2/main/sam2_configs/sam2_hiera_s.yaml

%cd /content
!git clone https://github.com/DepthAnything/Depth-Anything-V2.git
%cd Depth-Anything-V2
!mkdir -p checkpoints
!wget -q -P checkpoints https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth

In [ ]:
%cd /content/PaperTheater_ai_Pipeline

import os
import io
import json
import inspect
import importlib
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

from google import genai
from google.genai import types

from src.config import Paths, PipelineConfig
from src.io_utils import ensure_dirs, load_image, save_mask
from src.florence_parser import FlorenceParser
from src.sam2_segmenter import SAM2Segmenter
from src.depth_anything_runner import DepthRunner
from src.occlusion_heuristic import heuristic_complete
from src.mask_cleanup import cleanup_mask

import src.scene_builder
importlib.reload(src.scene_builder)
from src.scene_builder import (
    parse_florence_boxes,
    merge_segmented_by_label,
    build_stable_merged_objects,
)

import src.scene_representation
importlib.reload(src.scene_representation)
from src.scene_representation import build_scene_representation, save_scene_representation

import src.layer_planner
importlib.reload(src.layer_planner)
from src.layer_planner import plan_layers_deterministic

import src.layer_renderer
importlib.reload(src.layer_renderer)
from src.layer_renderer import build_object_mask_map

import src.layer_context_builder
importlib.reload(src.layer_context_builder)
from src.layer_context_builder import build_layer_contexts

paths = Paths()
cfg = PipelineConfig()
ensure_dirs(paths)

gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [ ]:
SCENE_NAME = "scene_2"
image_path = f"/content/PaperTheater_ai_Pipeline/data/input/{SCENE_NAME}.jpg"

SCENE_REPR_PATH = f"/content/PaperTheater_ai_Pipeline/data/intermediate/{SCENE_NAME}_scene_repr.json"
LAYER_PLAN_PATH = f"/content/PaperTheater_ai_Pipeline/data/intermediate/{SCENE_NAME}_layer_plan.json"
FULLFRAME_OUTPUT_DIR = f"/content/PaperTheater_ai_Pipeline/data/intermediate/{SCENE_NAME}_fullframe_prompt_normalized_gemini"

GEMINI_MODEL = "gemini-3.1-flash-image-preview"

STYLE_NORMALIZATION_PROMPT = """
Render this as a stylized Japanese paper theater / kamishibai layer.

GLOBAL STYLE RULES:
- flat paper-cut illustration
- simplified silhouettes and clean edges
- muted earthy palette
- minimal texture
- no photorealism
- no realistic skin or fur rendering
- no detailed lighting or harsh shadows
- consistent abstraction across all layers
- all layers should look like they were made by the same artist
- preserve the original composition and scale inside the full canvas
- output should feel like a clean cuttable layer for paper theater production
""".strip()

image = load_image(image_path, max_side=cfg.image_max_side)

plt.figure(figsize=(8, 12))
plt.imshow(image)
plt.axis("off")
plt.title(SCENE_NAME)
plt.show()

print("image shape:", image.shape)
print("output dir:", FULLFRAME_OUTPUT_DIR)
print("gemini model:", GEMINI_MODEL)

In [ ]:
florence = FlorenceParser(cfg.florence_model)
segmenter = SAM2Segmenter(cfg.sam2_config, cfg.sam2_checkpoint)
depth_runner = DepthRunner(cfg.depth_encoder)

caption = florence.get_dense_caption(image)
print(caption)

queries = [
    "a person",
    "a deer",
    "trees",
    "a building",
    "sky",
]

all_boxes = []

for q in queries:
    det_q = florence.get_open_vocab_detection(image, q)
    boxes_q = parse_florence_boxes(det_q, image.shape)

    print("\nQUERY:", q)
    print("RAW:", det_q)
    print("PARSED:", boxes_q)

    all_boxes.extend(boxes_q)

seen = set()
boxes = []
for b in all_boxes:
    key = (b["label"], tuple(b["bbox"]))
    if key not in seen:
        seen.add(key)
        boxes.append(b)

print("\nFINAL BOXES:")
print(boxes)
print("num boxes:", len(boxes))

fig, ax = plt.subplots(figsize=(8, 12))
ax.imshow(image)

for b in boxes:
    x1, y1, x2, y2 = b["bbox"]
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=2, edgecolor="red", facecolor="none"
    )
    ax.add_patch(rect)
    ax.text(x1, y1 - 5, b["label"], color="yellow", fontsize=10, backgroundcolor="black")

ax.axis("off")
plt.show()

segmented = segmenter.segment_boxes(image, boxes)
merged_segmented = merge_segmented_by_label(segmented)

depth = depth_runner.infer(image)
stable_objects = build_stable_merged_objects(merged_segmented, depth)

print("num boxes:", len(boxes))
print("num segmented:", len(segmented))
print("num merged_segmented:", len(merged_segmented))
print("num stable_objects:", len(stable_objects))

for i, s in enumerate(segmented):
    plt.figure(figsize=(6, 8))
    plt.imshow(image)
    plt.imshow(s["mask"], alpha=0.4)
    plt.title(f"{i}: {s['label']}")
    plt.axis("off")
    plt.show()

scene_repr = build_scene_representation(
    image_path=image_path,
    image_shape=image.shape,
    caption=caption,
    stable_objects=stable_objects,
)

save_scene_representation(scene_repr, SCENE_REPR_PATH)

layer_plan = plan_layers_deterministic(scene_repr)

with open(LAYER_PLAN_PATH, "w", encoding="utf-8") as f:
    json.dump(layer_plan, f, indent=2)

print("scene_repr saved to:", SCENE_REPR_PATH)
print("layer_plan saved to:", LAYER_PLAN_PATH)
print(layer_plan)

mask_records = []

for obj in stable_objects:
    mask = obj["mask"]
    label = obj["label"]
    cleaned_mask = cleanup_mask(
        heuristic_complete(mask, label),
        cfg.min_component_area,
        cfg.smooth_kernel
    )

    out_mask = paths.completed_dir / f"{SCENE_NAME}_mask_{obj['id']}.png"
    save_mask(out_mask, cleaned_mask)

    mask_records.append({
        "id": obj["id"],
        "label": label,
        "bbox": obj["bbox"],
        "mask_path": str(out_mask)
    })

print(mask_records)

object_mask_map = build_object_mask_map(mask_records)

layer_contexts = build_layer_contexts(
    scene_repr=scene_repr,
    layer_plan=layer_plan,
    object_mask_map=object_mask_map,
)

for ctx in layer_contexts:
    print(
        ctx["order"],
        ctx["layer_name"],
        ctx["object_ids"],
        "| labels =", ctx["labels"],
        "| ownership_bbox =", ctx["ownership_bbox"],
        "| visible_bbox =", ctx["visible_bbox"],
        "| front =", ctx["front_layer_names"],
    )

In [ ]:
def apply_focus_mask_fullframe(image_rgb, ownership_mask, front_occlusion_mask):
    focused = image_rgb.copy()

    ownership = ownership_mask > 0
    front = front_occlusion_mask > 0

    focused[~ownership] = 0
    focused[front] = 0

    return focused


def mask_to_rgba_fullframe(image_rgb, mask):
    alpha = (mask.astype(np.uint8)) * 255
    return np.dstack([image_rgb, alpha])


def save_bool_mask(mask, path):
    Image.fromarray((mask.astype(np.uint8) * 255)).save(path)


def build_fullframe_prompt(layer_context):
    label = layer_context["labels"][0] if layer_context["labels"] else layer_context["layer_name"]

    return f"""
Generate only the {label} layer for a layered paper-theater scene.

Scene context:
{layer_context["scene_caption"]}

Target layer:
- layer name: {layer_context["layer_name"]}
- semantic label: {label}

{STYLE_NORMALIZATION_PROMPT}

IMPORTANT RULES:
1. Generate only this layer.
2. Keep the layer in its correct position within the full image.
3. Do not add objects from other layers.
4. Respect the ownership mask region and overall scene placement.
5. Do not redesign the whole scene.
6. Treat blacked-out or masked-out regions as occlusion or simple unobtrusive support.
7. The result must behave as a clean full-frame cuttable layer.
8. Maintain the same visual language used for other layers.
9. Keep colors and abstraction level consistent across the whole project.
10. Output must align to the original full image canvas.
11. Keep non-target regions plain and unobtrusive.
12. Do not hallucinate unrelated scene elements.

OUTPUT GOAL:
A full-frame layer image that preserves alignment with the source scene while normalizing the rendered subject into a cohesive kamishibai paper-cut aesthetic.
""".strip()


def extract_first_image(response):
    # Gemini can return text and image parts together.
    if hasattr(response, "parts") and response.parts is not None:
        for part in response.parts:
            if getattr(part, "inline_data", None) is not None:
                return part.as_image()

    if hasattr(response, "candidates") and response.candidates:
        for cand in response.candidates:
            content = getattr(cand, "content", None)
            parts = getattr(content, "parts", None) if content is not None else None
            if parts:
                for part in parts:
                    if getattr(part, "inline_data", None) is not None:
                        return part.as_image()

    raise RuntimeError("Gemini response did not include an image part.")


def gemini_edit(image_rgb, prompt, model_name=GEMINI_MODEL):
    input_image = Image.fromarray(image_rgb.astype(np.uint8))
    orig_h, orig_w = image_rgb.shape[:2]

    response = gemini_client.models.generate_content(
        model=model_name,
        contents=[input_image, prompt],
        config=types.GenerateContentConfig(
            response_modalities=["TEXT", "IMAGE"]
        ),
    )

    out_img = extract_first_image(response).convert("RGB")
    if out_img.size != (orig_w, orig_h):
        out_img = out_img.resize((orig_w, orig_h), Image.Resampling.LANCZOS)

    return np.array(out_img)

# Optional preview
if layer_contexts:
    preview_prompt = build_fullframe_prompt(layer_contexts[0])
    print(preview_prompt[:2500])

In [ ]:
def realize_single_layer_fullframe(
    image,
    layer_context,
    output_dir,
    model_name,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    order = layer_context["order"]
    layer_name = layer_context["layer_name"]

    ownership_mask = layer_context["ownership_mask"] > 0
    front_occlusion_mask = layer_context["front_occlusion_mask"] > 0
    visible_mask = layer_context["visible_mask"] > 0

    focused_input = apply_focus_mask_fullframe(image, ownership_mask, front_occlusion_mask)
    prompt = build_fullframe_prompt(layer_context)

    generated_full = gemini_edit(
        focused_input,
        prompt=prompt,
        model_name=model_name,
    )

    input_focus_path = output_dir / f"{order:02d}_{layer_name}_input_focus_full.png"
    ownership_mask_path = output_dir / f"{order:02d}_{layer_name}_ownership_mask_full.png"
    front_mask_path = output_dir / f"{order:02d}_{layer_name}_front_occlusion_mask_full.png"
    visible_mask_path = output_dir / f"{order:02d}_{layer_name}_visible_mask_full.png"
    prompt_path = output_dir / f"{order:02d}_{layer_name}_prompt.txt"
    generated_full_path = output_dir / f"{order:02d}_{layer_name}_generated_full.png"
    generated_visible_full_path = output_dir / f"{order:02d}_{layer_name}_generated_visible_full.png"
    original_visible_full_path = output_dir / f"{order:02d}_{layer_name}_original_visible_full.png"

    Image.fromarray(focused_input).save(input_focus_path)
    save_bool_mask(ownership_mask, ownership_mask_path)
    save_bool_mask(front_occlusion_mask, front_mask_path)
    save_bool_mask(visible_mask, visible_mask_path)

    with open(prompt_path, "w", encoding="utf-8") as f:
        f.write(prompt)

    Image.fromarray(generated_full).save(generated_full_path)

    generated_visible_rgba = mask_to_rgba_fullframe(generated_full, visible_mask)
    original_visible_rgba = mask_to_rgba_fullframe(image, visible_mask)

    Image.fromarray(generated_visible_rgba).save(generated_visible_full_path)
    Image.fromarray(original_visible_rgba).save(original_visible_full_path)

    return {
        "layer_name": layer_name,
        "order": order,
        "input_focus_full_path": str(input_focus_path),
        "ownership_mask_full_path": str(ownership_mask_path),
        "front_occlusion_mask_full_path": str(front_mask_path),
        "visible_mask_full_path": str(visible_mask_path),
        "prompt_path": str(prompt_path),
        "generated_full_path": str(generated_full_path),
        "generated_visible_full_path": str(generated_visible_full_path),
        "original_visible_full_path": str(original_visible_full_path),
    }

fullframe_results = []

for ctx in layer_contexts:
    print("\n==============================")
    print("Processing full-frame layer:", ctx["layer_name"])
    print("==============================")

    result = realize_single_layer_fullframe(
        image=image,
        layer_context=ctx,
        output_dir=FULLFRAME_OUTPUT_DIR,
        model_name=GEMINI_MODEL,
    )

    fullframe_results.append(result)

    print("Saved:")
    print("  generated_full :", result["generated_full_path"])
    print("  generated_alpha:", result["generated_visible_full_path"])

In [ ]:
for result in fullframe_results:
    print("\n======================")
    print("Layer:", result["layer_name"])
    print("======================")

    paths_to_show = [
        ("generated_full", result["generated_full_path"]),
        ("generated_visible_full", result["generated_visible_full_path"]),
        ("original_visible_full", result["original_visible_full_path"]),
    ]

    for name, p in paths_to_show:
        img = Image.open(p)

        plt.figure(figsize=(8, 10))
        plt.imshow(img)
        plt.title(f"{result['layer_name']} — {name}")
        plt.axis("off")
        plt.show()

print("num full-frame results:", len(fullframe_results))

## Notes

- This notebook intentionally keeps the rest of your working repo pipeline intact.
- The main change is swapping the realization step to Gemini's image editing flow by sending the focused full-frame image plus the per-layer prompt.
- Gemini docs show image editing by passing the source image together with a text instruction to `generate_content`, using models like `gemini-3.1-flash-image-preview`. citeturn241549view0turn241549view1
- The Google Gen AI Python SDK is installed via `google-genai`, and the client can be initialized with `genai.Client(api_key=...)`. citeturn862909view1
